In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from utils import bkv_nkv_from_verse_id, check_and_create_file
from constants import TMP_PATH

# Enable tqdm for pandas
tqdm.pandas()

In [2]:
na28_df = pd.read_csv("../na28-crawler/na28_verses.csv")
na28_df.dropna(subset=["text"], inplace=True)

In [3]:
ecm_df = pd.read_csv("../ecm-crawler/ecm_verses.csv")
ecm_df.dropna(subset=["text"], inplace=True)

In [4]:
na28_df["ga"] = "na28"
ecm_df["ga"] = "ecm"

verses_df = pd.concat([na28_df, ecm_df])

In [ ]:
verses_df = verses_df.progress_apply(bkv_nkv_from_verse_id, axis=1, verse_id_col="nkv")

In [6]:
import re
import unicodedata


def str_remove_diacritics(s: str) -> str:
    """Normalize string by removing accents and converting to lower case.

    - unicodedata.normalize('NFKD', s): normalizes the input Unicode string s using NFKD normalization. Valid normalization forms are 'NFC', 'NFKC', 'NFD', and 'NFKD'.
    - (c for c in ...): This is a generator expression that iterates over each character c in the normalized string obtained in the previous step.
    - if unicodedata.category(c) != 'Mn': This condition checks whether the Unicode character c belongs to the
        category 'Mn' (Mark, Non-Spacing). Characters in this category are combining characters that modify the
        meaning of the preceding base character. The condition filters out all combining characters from the
        normalized string.

    :param s: String to be normalized.
    :return:  normalized string
    """
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if unicodedata.category(c) != "Mn"
    )


def str_remove_punctuation(s: str) -> str:
    """Strip punctuation from input string

    :param s: string to strip punctuation from
    :return: string without punctuation
    """
    # remove everything expect characters, spaces, '[' and ']'
    s = re.sub(r"[^\w\s\[\]]", "", s)
    # normalize by removing multiple consecutive whitespaces
    s = re.sub(r"\s+", " ", s)
    # remove leading/trailing whitespaces
    return s.strip()

In [ ]:
verses_df["text"] = verses_df["text"].progress_apply(lambda x: str_remove_diacritics(x))
verses_df["text"] = verses_df["text"].progress_apply(
    lambda x: str_remove_punctuation(x)
)

In [ ]:
check_and_create_file(TMP_PATH + "na28ecm_verses.csv")
verses_df.to_csv(
    TMP_PATH + "na28ecm_verses.csv", mode="w", encoding="utf8", index=False
)